## Installing Dependencies 

In [1]:
pip install -q langgraph langchain-ollama fastmcp httpx nest_asyncio pydantic

Note: you may need to restart the kernel to use updated packages.


## Initialize FastMCP Server

In [1]:
import nest_asyncio
from fastmcp import FastMCP, Client

nest_asyncio.apply()

# 1. Initialize FastMCP Server
mcp = FastMCP("Ecommerce-Compliance-Server")

# Tool 1: Banned Words Inspector
@mcp.tool
def evaluate_brand_compliance(copy_text: str) -> dict:
    """Evaluates marketing ad copy against compliance and brand safety guidelines."""
    banned_words = ["cheap", "guaranteed", "100% free", "miracle"]
    found = [word for word in banned_words if word in copy_text.lower()]
    return {
        "is_valid": len(found) == 0,
        "violations": found,
        "character_count": len(copy_text)
    }

# Tool 2: Character Limit Inspector (NEW TOOL)
@mcp.tool
def check_length_and_formatting(copy_text: str, max_chars: int = 100) -> dict:
    """Checks if copy fits social media length constraints and uses proper punctuation."""
    char_len = len(copy_text)
    is_valid = char_len <= max_chars and copy_text.endswith((".", "!", "?"))
    
    issues = []
    if char_len > max_chars:
        issues.append(f"Too long ({char_len}/{max_chars} chars)")
    if not copy_text.endswith((".", "!", "?")):
        issues.append("Missing ending punctuation")
        
    return {
        "is_valid": is_valid,
        "issues": issues,
        "char_count": char_len
    }

# Connect In-Memory Client
mcp_client = Client(mcp)
print("✅ FastMCP Server with 2 Tools Ready")

✅ FastMCP Server with 2 Tools Ready


## Build the LangGraph Agent Workflow

In [2]:
from typing import TypedDict, List
from langgraph.graph import StateGraph, END
from langchain_ollama import ChatOllama

# 1. Updated State Schema to capture state transition logs
class AgentState(TypedDict):
    product_name: str
    draft_copy: str
    feedback: str
    iterations: int
    status: str
    transition_logs: List[str]  # NEW: Tracks history across state updates

# Initialize Local LLM
llm = ChatOllama(model="qwen2.5:3b", temperature=0.2, repeat_penalty=1.2)

# Node A: Generator
def generate_copy_node(state: AgentState):
    feedback = state.get("feedback", "")
    if feedback:
        prompt = (
            f"Task: Write a short 1-sentence ad for: {state['product_name']}.\n"
            f"REVISION NEEDED: '{feedback}'. Keep under 100 characters."
        )
    else:
        prompt = f"Task: Write a single short 1-sentence ad for: {state['product_name']}."

    response = llm.invoke(prompt)
    new_draft = response.content.strip()
    
    # Log node execution to state
    logs = state.get("transition_logs", [])
    logs.append(f"[Node: Generator] Produced draft (Iteration {state.get('iterations', 0) + 1}): '{new_draft}'")
    
    return {
        "draft_copy": new_draft,
        "iterations": state.get("iterations", 0) + 1,
        "transition_logs": logs
    }

# Node B: Evaluator calling BOTH FastMCP Tools
async def evaluate_copy_node(state: AgentState):
    async with mcp_client:
        # Call Tool 1: Compliance
        res1 = await mcp_client.call_tool("evaluate_brand_compliance", {"copy_text": state["draft_copy"]})
        data1 = res1.data if hasattr(res1, "data") else res1
        
        # Call Tool 2: Length/Formatting
        res2 = await mcp_client.call_tool("check_length_and_formatting", {"copy_text": state["draft_copy"], "max_chars": 100})
        data2 = res2.data if hasattr(res2, "data") else res2

    # Aggregate feedback from both tools
    issues = []
    if not data1.get("is_valid"):
        issues.append(f"Forbidden words: {', '.join(data1.get('violations', []))}")
    if not data2.get("is_valid"):
        issues.append(f"Formatting issues: {', '.join(data2.get('issues', []))}")

    logs = state.get("transition_logs", [])
    if not issues:
        logs.append("[Node: Evaluator] ✅ PASSED all checks (Compliance + Formatting).")
        return {"status": "APPROVED", "feedback": "", "transition_logs": logs}
    else:
        combined_feedback = " | ".join(issues)
        logs.append(f"[Node: Evaluator] ❌ REJECTED -> {combined_feedback}")
        return {"status": "REJECTED", "feedback": combined_feedback, "transition_logs": logs}

# Routing logic remains unchanged
def routing_logic(state: AgentState):
    if state["status"] == "APPROVED":
        return "approved_end"
    elif state["iterations"] >= 3:
        return "max_iterations_reached"
    return "try_again"

# Build Graph
workflow = StateGraph(AgentState)
workflow.add_node("generator", generate_copy_node)
workflow.add_node("evaluator", evaluate_copy_node)

workflow.set_entry_point("generator")
workflow.add_edge("generator", "evaluator")
workflow.add_conditional_edges("evaluator", routing_logic, {
    "approved_end": END,
    "max_iterations_reached": END,
    "try_again": "generator"
})

agent = workflow.compile()

## Inspecting Transition Logs

In [3]:
input_data = {
    "product_name": "Leather Shoes (guaranteed free delivery!)",
    "iterations": 0,
    "transition_logs": []
}

# Stream state changes step-by-step
async for event in agent.astream(input_data):
    for node_name, state_update in event.items():
        print(f"=== STATE TRANSITION: Node [{node_name}] ===")
        # Print latest log entry from the state dictionary
        if "transition_logs" in state_update:
            print(f"LOG: {state_update['transition_logs'][-1]}\n")

=== STATE TRANSITION: Node [generator] ===
LOG: [Node: Generator] Produced draft (Iteration 1): 'Get stylish and comfortable with our premium leather shoes, now available with guaranteed free shipping!'

=== STATE TRANSITION: Node [evaluator] ===
LOG: [Node: Evaluator] ❌ REJECTED -> Forbidden words: guaranteed | Formatting issues: Too long (104/100 chars)

=== STATE TRANSITION: Node [generator] ===
LOG: [Node: Generator] Produced draft (Iteration 2): 'Luxury leather shoes, enjoy free shipping now!'

=== STATE TRANSITION: Node [evaluator] ===
LOG: [Node: Evaluator] ✅ PASSED all checks (Compliance + Formatting).



## Running and Streaming the Agent Loop

In [4]:
# Initial Input State
input_data = {
    "product_name": "Premium Handmade Leather Boots (guaranteed 100% free delivery)",
    "iterations": 0
}

print("🚀 Starting Agentic Generation Loop...\n")

# Run async streaming natively in Jupyter using 'await'
async for output in agent.astream(input_data):
    for node_name, state_update in output.items():
        print(f"--- Node Executed: [{node_name}] ---")
        if "draft_copy" in state_update:
            print(f"  📝 Draft Produced: \"{state_update['draft_copy']}\"")
        if "status" in state_update:
            print(f"  🔍 Status: {state_update['status']}")
            if state_update['feedback']:
                print(f"  ⚠️  Inspector Feedback: {state_update['feedback']}")
        print()

🚀 Starting Agentic Generation Loop...

--- Node Executed: [generator] ---
  📝 Draft Produced: "Experience unparalleled comfort and durability with our premium handmade leather boots, including complimentary free shipping worldwide!"

--- Node Executed: [evaluator] ---
  🔍 Status: REJECTED
  ⚠️  Inspector Feedback: Formatting issues: Too long (136/100 chars)

--- Node Executed: [generator] ---
  📝 Draft Produced: "Premium handmade leather boots, guaranteed 100% free delivery!"

--- Node Executed: [evaluator] ---
  🔍 Status: REJECTED
  ⚠️  Inspector Feedback: Forbidden words: guaranteed, 100% free

--- Node Executed: [generator] ---
  📝 Draft Produced: "Elevate your style with premium handmade leather boots - FREE shipping this winter!"

--- Node Executed: [evaluator] ---
  🔍 Status: APPROVED

